# RUKOPYS Full Validation Diagnostic

Measures the frozen baseline on all 143 validation pages. This notebook does not train or modify model weights. All stages save one JSONL record per completed page and resume automatically.

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path

ROOT = Path('/kaggle/working/rukopys')
REPO = 'https://github.com/thanhbinh55/rukopys.git'

def run(command):
    print('$', command)
    return subprocess.run(command, shell=True, executable='/bin/bash', check=True)

if ROOT.exists():
    run(f'git -C {ROOT} fetch origin main')
    run(f'git -C {ROOT} reset --hard origin/main')
else:
    run(f'git clone {REPO} {ROOT}')

run(f'''{sys.executable} -m pip install -q -U pip wheel setuptools packaging
{sys.executable} -m pip install -q 'transformers==4.57.1' 'peft==0.17.1' 'accelerate>=1.0.0,<2.0.0' qwen-vl-utils timm bitsandbytes ultralytics pandas numpy pillow rapidfuzz pyyaml''')
os.chdir(ROOT)
run('git rev-parse HEAD')

In [ ]:
os.environ.update({
    'PROJECT_ROOT': str(ROOT),
    'HTR_ART_DIR': str(ROOT / 'local_working' / 'htr_artifacts'),
    'HF_HOME': '/tmp/hf_cache',
    'TRANSFORMERS_CACHE': '/tmp/hf_cache',
    'YOLO_CONFIG_DIR': '/tmp/Ultralytics',
    'PYTHONPATH': f'{ROOT}:{ROOT / "scripts"}',
    'TOKENIZERS_PARALLELISM': 'false',
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True,max_split_size_mb:128',
    'CUDA_VISIBLE_DEVICES': '0,1',
    'USE_FLASH_ATTN': '0',
    'QLORA_4BIT': '1',
    'GPU_MAX_MEMORY': '13GiB',
    'CPU_MAX_MEMORY': '24GiB',
    'MIN_PIXELS': str(128 * 28 * 28),
    'MAX_PIXELS': str(384 * 28 * 28),
    'MAX_TOKENS': '96',
    'OCR_BATCH': '1',
})
Path('/tmp/hf_cache').mkdir(parents=True, exist_ok=True)
run('python scripts/09_prepare_kaggle_diagnostic.py')
run('nvidia-smi')

In [ ]:
run('''python scripts/08_full_validation_diagnostic.py \\
  --modes detector,gt_ocr,e2e,report \\
  --conf 0.20 \\
  --iou 0.45 \\
  --imgsz 1024 \\
  --ocr-batch 1 \\
  --save-every 1 \\
  --time-budget-hours 10.5''')

In [ ]:
import json
OUT = Path('/kaggle/working/full_validation_diagnostic')
report = json.loads((OUT / 'full_validation_report.json').read_text())
print(json.dumps(report['completed_pages'], indent=2))
print((OUT / 'full_validation_report.md').read_text()[:12000])
run('tar -czf /kaggle/working/full_validation_diagnostic.tgz -C /kaggle/working full_validation_diagnostic')
run('ls -lh /kaggle/working/full_validation_diagnostic.tgz')